# Gráficas Test 2 ($S_n$) + Comparativas $T_n$ vs $S_n$

Carga los CSV generados por `02_run_test2.ipynb` y genera todas las gráficas. Si además existe el CSV summary de $T_n$ (`tn_simulation_summary.csv`), genera las comparativas cruzadas.

Cambia `use_quick = True` si solo tienes los datos quick.

In [ ]:
# --- Setup ---
import sys, os
from pathlib import Path

ROOT = Path.cwd().parent if (Path.cwd().name == 'notebooks') else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src.plotting_sn import (
    plot_type_i_error_sn, plot_power_curves_sn, plot_runtime_sn,
    plot_q1_vs_q2, plot_weights_compare, plot_power_heatmap,
    plot_pvalue_distribution_h0_sn,
    plot_tn_vs_sn_power, plot_tn_vs_sn_runtime,
)

print(f'Working dir: {Path.cwd()}')

## Cargar CSVs

In [ ]:
use_quick = False
suffix = '_quick' if use_quick else ''

raw_csv = ROOT / 'results' / 'data' / f'sn_simulation_raw{suffix}.csv'
sum_csv = ROOT / 'results' / 'data' / f'sn_simulation_summary{suffix}.csv'

if not raw_csv.exists():
    raise FileNotFoundError(
        f'No se encontró {raw_csv}. Ejecuta primero `02_run_test2.ipynb`.'
    )

df = pd.read_csv(raw_csv)
summary = pd.read_csv(sum_csv)
print(f'S_n: {raw_csv.name} ({len(df):,} filas) / {sum_csv.name} ({len(summary)} filas)')
print(f'q: {sorted(df.q.unique())}, pesos: {sorted(df.weight.unique())}')
print(f'n: {sorted(df.n.unique())}, estimadores: {sorted(df.estimator.unique())}')

# Carga T_n si existe para comparativas
tn_sum_csv = ROOT / 'results' / 'data' / 'tn_simulation_summary.csv'
summary_tn = None
if tn_sum_csv.exists():
    summary_tn = pd.read_csv(tn_sum_csv)
    print(f'\nT_n: {tn_sum_csv.name} ({len(summary_tn)} filas) — comparativas disponibles')
else:
    print(f'\n(no se encontró {tn_sum_csv.name} — saltando comparativas T_n vs S_n)')

## Resumen pivotado

In [ ]:
print('Tasa de rechazo (argmin, q=2, columnas = peso):')
sub = summary[(summary['estimator']=='argmin') & (summary['q']==2)]
if not sub.empty:
    display(sub.pivot_table(index=['dist','n'], columns='weight', values='reject_rate').round(3))

print('\nTiempo medio por test (s), por estimador y q (promedio sobre pesos):')
display(summary.pivot_table(index=['estimator','n'], columns='q', values='mean_time_s').round(3))

In [ ]:
fig_dir = ROOT / 'results' / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)
alpha = 0.05

def show(path):
    if path is not None and Path(path).exists():
        display(Image(filename=str(path)))
    else:
        print('(sin imagen)')

def show_many(paths):
    for p in paths:
        show(p)

### Error Tipo I bajo H₀ (una figura por (q, w))

In [ ]:
show_many(plot_type_i_error_sn(summary, alpha=alpha, outdir=fig_dir))

### Curvas de potencia bajo Hₐ (una figura por (q, w))

In [ ]:
show_many(plot_power_curves_sn(summary, outdir=fig_dir))

### Comparación q=1 vs q=2

In [ ]:
show(plot_q1_vs_q2(summary, outdir=fig_dir))

### Comparación entre funciones de peso w(t)

In [ ]:
show(plot_weights_compare(summary, outdir=fig_dir))

### Heatmap de potencia (estimador argmin)

In [ ]:
show(plot_power_heatmap(summary, outdir=fig_dir))

### Tiempo de ejecución vs n

In [ ]:
show(plot_runtime_sn(summary, outdir=fig_dir))

### Distribución del p-valor bajo H₀

In [ ]:
show(plot_pvalue_distribution_h0_sn(df, outdir=fig_dir))

## Comparativas T_n vs S_n

Solo si `tn_simulation_summary.csv` existe.

In [ ]:
if summary_tn is not None:
    print('Potencia:')
    show(plot_tn_vs_sn_power(summary_tn, summary, outdir=fig_dir))
    print('Runtime:')
    show(plot_tn_vs_sn_runtime(summary_tn, summary, outdir=fig_dir))
else:
    print('No hay datos de T_n para comparar. Ejecuta primero el Test 1.')

## Exploración interactiva

Plantilla para hacer cortes propios.

In [ ]:
# Ejemplo: para una distribución dada, comparar todas las (q, w) combinations
dist = 'Gamma(k=2.0,s=1.0)'
est = 'argmin'

sub = summary[(summary['dist']==dist) & (summary['estimator']==est)].copy()
sub['config'] = 'q=' + sub['q'].astype(str) + ' | ' + sub['weight']

fig, ax = plt.subplots(figsize=(7,4))
for cfg in sorted(sub['config'].unique()):
    s2 = sub[sub['config']==cfg].sort_values('n')
    ax.errorbar(s2['n'], s2['reject_rate'], yerr=2*s2['se_rate'],
                marker='o', capsize=3, label=cfg)
ax.set_xscale('log')
ax.set_xlabel('n (log)')
ax.set_ylabel('Potencia empírica')
ax.set_title(f'{dist} | {est}: efecto de (q, w)')
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=8)
plt.show()